# ベンチマークの評価データが学習に漏れていないかを調べる (データ汚染の検出)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/benchmark_contamination.ipynb)

NLP クックブック 12-10 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

公開ベンチマークで LLM を比べるとき、評価データの問題をモデルが学習時に見ていた (データ汚染) かどうかを調べたい。問題文の続きを生成させる方法、尤度にもとづく方法 (Min-K% Prob など)、言い換えや選択肢の並べ替えで正解率が落ちるかを見る方法を試し、どこまで分かるかを確かめる。

In [1]:
!pip install -q datasets

In [2]:
import gc, itertools, json, re, time, zlib
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from scipy import stats
from sklearn.metrics import roc_auc_score
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
rng = np.random.default_rng(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16

# JGLUE (Kurihara et al., 2022) の JCommonsenseQA (5 択の常識問題)。CC BY-SA 4.0。この章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」と同じデータ
val = [dict(ex) for ex in load_dataset("sbintuitions/JCommonsenseQA", split="validation").shuffle(seed=0).select(range(300))]
train = [dict(ex) for ex in load_dataset("sbintuitions/JCommonsenseQA", split="train").shuffle(seed=0).select(range(600))]
SEEN, UNSEEN = np.arange(150), np.arange(150, 300)   # 後半の実験で、評価データの前半 150 問だけをわざと学習させる
choices = lambda ex: [ex[f"choice{i}"] for i in range(5)]

def load_tokenizer(name, **kwargs):
    tokenizer = AutoTokenizer.from_pretrained(name, **kwargs)
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def mc_text(question, opts, label=None):
    """素のテキストの問題の形式。label を渡すと答えの番号まで付ける (学習用)"""
    text = f"質問: {question}\n選択肢: " + " ".join(f"{i + 1}. {c}" for i, c in enumerate(opts)) + "\n答え:"
    return text if label is None else text + str(label + 1)

# 選択肢の並べ替え: 正解の位置が必ず変わるように並べ替える (正解の番号を覚えているだけなら、ここで外れる)
shuffled = []
for ex in val:
    while True:
        perm = rng.permutation(5)
        if perm[ex["label"]] != ex["label"]:
            break
    opts = [None] * 5
    for i, j in enumerate(perm):
        opts[j] = ex[f"choice{i}"]
    shuffled.append((opts, int(perm[ex["label"]])))
print(mc_text(val[0]["question"], choices(val[0]), val[0]["label"]))
print(mc_text(val[0]["question"], shuffled[0][0], shuffled[0][1]))

質問: 福井県の南西部、若狭のほぼ中央に位置する市は？
選択肢: 1. 小鍋 2. 小浜 3. 大浜 4. 小売 5. 高浜
答え:2
質問: 福井県の南西部、若狭のほぼ中央に位置する市は？
選択肢: 1. 小売 2. 高浜 3. 小鍋 4. 大浜 5. 小浜
答え:5


In [3]:
# 共通の道具: 文のトークンごとの対数確率と、答えの番号の予測
def batches(n, bs):
    return [range(i, min(i + bs, n)) for i in range(0, n, bs)]

@torch.no_grad()
def token_logprobs(model, tok, texts, bs=32):
    """各文について、2 番目以降のトークンの対数確率の配列を返す (文頭に BOS を付ける)。
    追加学習の文には BOS を付けないので、最初のトークンの条件だけが学習時と少し違う (影響はその 1 トークンだけ)"""
    out = []
    for b in batches(len(texts), bs):
        seqs = [([tok.bos_token_id] if tok.bos_token_id is not None else []) + tok(texts[i], add_special_tokens=False)["input_ids"] for i in b]
        L = max(map(len, seqs))
        ids = torch.tensor([s + [tok.pad_token_id] * (L - len(s)) for s in seqs], device=model.device)       # 右パディング
        mask = (torch.arange(L, device=model.device) < torch.tensor([len(s) for s in seqs], device=model.device)[:, None]).long()
        lp = model(ids, attention_mask=mask).logits[:, :-1].float().log_softmax(-1).gather(2, ids[:, 1:, None])[..., 0]
        out += [lp[k, :len(s) - 1].cpu().numpy() for k, s in enumerate(seqs)]
    return out

@torch.no_grad()
def predict_number(model, tok, prompts, bs=32):
    """「答え:」の次のトークンとして 1〜5 のどれの確率が高いか"""
    digits = [tok(str(i), add_special_tokens=False)["input_ids"][-1] for i in range(1, 6)]
    tok.padding_side = "left"
    preds = []
    for b in batches(len(prompts), bs):
        enc = tok([prompts[i] for i in b], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)           # 左パディングでも位置を 0 から数える
        preds += model(**enc, position_ids=pos, logits_to_keep=1).logits[:, -1, digits].argmax(-1).tolist()
    tok.padding_side = "right"
    return np.array(preds)

# 選択肢の並び順の検定 (Oren et al. (2023) の「元の並び順は、並べ替えた順より尤度が高いか」という考え方を、選択肢の順序に当てはめたもの)。
# 汚染がなければ、元の並び順は 10 通りの並べ替えと区別がつかず、順位 (並べ替えより尤度が高い割合) の平均は 0.5 になるはず
amp = lambda: torch.autocast("cuda", dtype=DTYPE)
def as_json(ex, opts):     # JGLUE の配布ファイル (JSON Lines) に似せた形式
    return json.dumps({"q_id": ex["q_id"], "question": ex["question"], **{f"choice{i}": c for i, c in enumerate(opts)}}, ensure_ascii=False)
FORMATS = {"素のテキスト": lambda ex, opts: mc_text(ex["question"], opts), "JSON": as_json}
ALL_PERMS = list(itertools.permutations(range(5)))[1:]          # 元の順 (0, 1, 2, 3, 4) を除く 119 通り
PERMS = [ALL_PERMS[i] for i in np.random.default_rng(0).choice(len(ALL_PERMS), 10, replace=False)]   # 重複なく 10 通りを選ぶ

def order_ranks(model, tok, fmt, idx):
    texts = []
    for i in idx:
        texts += [fmt(val[i], choices(val[i]))] + [fmt(val[i], [val[i][f"choice{j}"] for j in p]) for p in PERMS]
    with amp():
        ll = np.array([lp.sum() for lp in token_logprobs(model, tok, texts)]).reshape(len(idx), 1 + len(PERMS))
    return (ll[:, :1] > ll[:, 1:]).mean(1) + 0.5 * (ll[:, :1] == ll[:, 1:]).mean(1)

def order_test(model, tok, fmt, idx):
    rank = order_ranks(model, tok, fmt, idx)
    p = stats.ttest_1samp(rank, 0.5, alternative="greater").pvalue          # 片側検定: 元の並び順が有利か
    return f"{rank.mean():.3f} (p={p:.2g}, 最も尤度が高い割合 {(rank == 1).mean():.0%})"   # 最後の値は、汚染がなければ 1/11 (約 9%) 程度

def paired_diff(a, b, n_boot=2000):
    """同じ問題で比べる対応のあるブートストラップ: 平均の差 (ポイント) と 95% 信頼区間"""
    idx = rng.integers(0, len(a), (n_boot, len(a)))
    d = (a[idx].mean(1) - b[idx].mean(1)) * 100
    return f"{(a.mean() - b.mean()) * 100:+5.1f} [{np.percentile(d, 2.5):+5.1f}, {np.percentile(d, 97.5):+5.1f}]"

In [4]:
# (1) 言い換えに強いか: 大きめのモデルに問題文を言い換えさせ、元の問題・言い換えた問題・選択肢を並べ替えた問題で正解率を比べる
#     対象はこの章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」の 3b (SB Intuitions、MIT ライセンス)。Colab の T4 で 2〜3 分
name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tok = load_tokenizer(name)
llm = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()
chat = lambda content: tok.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True, tokenize=False)

@torch.no_grad()
def generate(prompts, bs=32, max_new_tokens=60):
    tok.padding_side = "left"
    outs = []
    for b in batches(len(prompts), bs):
        enc = tok([prompts[i] for i in b], return_tensors="pt", padding=True, add_special_tokens=False).to(llm.device)
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return outs

PARA = "次の質問文を、意味と答えが変わらないように、別の言葉と語順で言い換えてください。言い換えた質問文だけを 1 行で出力してください。\n\n質問文: {}"
para = [o.strip().split("\n")[0].strip() or ex["question"] for o, ex in zip(generate([chat(PARA.format(ex["question"])) for ex in val]), val)]
for ex, p in list(zip(val, para))[:4]:
    print(f"{ex['question']}\n → {p}")
print("言い換えが元と同じだった問題:", sum(p == ex["question"] for p, ex in zip(para, val)), "/", len(val))

def mc_prompt(question, opts):   # この章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」と同じ 0-shot の指示
    listing = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(opts))
    return chat(f"次の質問の答えとして最も適切なものを選び、番号だけを答えてください。\n\n質問: {question}\n{listing}")

gold = np.array([ex["label"] for ex in val])
real = {"元の問題": predict_number(llm, tok, [mc_prompt(ex["question"], choices(ex)) for ex in val]) == gold,
        "問題文を言い換え": predict_number(llm, tok, [mc_prompt(p, choices(ex)) for p, ex in zip(para, val)]) == gold,
        "選択肢を並べ替え": predict_number(llm, tok, [mc_prompt(ex["question"], o) for ex, (o, _) in zip(val, shuffled)]) == np.array([l for _, l in shuffled])}
print({k: f"{v.mean():.1%}" for k, v in real.items()}, f"({len(val)} 問)")
print("言い換え − 元:", paired_diff(real["問題文を言い換え"], real["元の問題"]), "/ 並べ替え − 元:", paired_diff(real["選択肢を並べ替え"], real["元の問題"]))
del llm; gc.collect(); torch.cuda.empty_cache()

福井県の南西部、若狭のほぼ中央に位置する市は？
 → 福井県の南西部、若狭地域の中心部にある市はどこですか？
セブンイレブンやローソンといえば、どんなお店のこと？
 → セブンイレブンやローソンは、どのようなコンビニエンスストアを指すのでしょうか？
でっかい？
 → 大きいですか？
よくわかるように述べることは？
 → どのように明確に説明するかについて述べることは何ですか？
言い換えが元と同じだった問題: 0 / 300
{'元の問題': '87.7%', '問題文を言い換え': '86.3%', '選択肢を並べ替え': '89.0%'} (300 問)
言い換え − 元:  -1.3 [ -4.7,  +2.0] / 並べ替え − 元:  +1.3 [ -3.0,  +5.7]


In [5]:
# (2) 汚染をわざと起こす: 0.5b のモデルを、JCommonsenseQA の学習データ 600 問と、評価データの前半 150 問 (SEEN) で追加学習する。
#     後半 150 問 (UNSEEN) は学習させない。どちらが漏れたかは分かっているので、各検出法がそれを見分けられるかを確かめられる
#     (事前学習だけのモデルは番号で答える形式に従いにくいと考え、正解率の変化を見やすいように指示チューニング済みのモデルを使う。事前学習モデルの正解率はここでは測っていない)
name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"      # SB Intuitions、MIT ライセンス
tok = load_tokenizer(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32, device_map="cuda")    # 学習は float32 の重み + 混合精度 (学習するので GPU 1 枚に載せる)
for p in [*model.model.embed_tokens.parameters(), *model.lm_head.parameters()]:
    p.requires_grad = False                  # 語彙の埋め込みと出力層は固定する (T4 のメモリに収めるため)
def detect(model):
    """SEEN / UNSEEN の各問題について、検出のためのスコアを求める (どれも「大きいほど学習に含まれていたらしい」向き)"""
    model.eval()
    with amp():
        texts = [mc_text(ex["question"], choices(ex), ex["label"]) for ex in val]
        lps = token_logprobs(model, tok, texts)
        # 続きの生成: 質問と 1 番目の選択肢までを見せ、2〜5 番目の選択肢を貪欲法で書かせて、元の選択肢と一致した数を数える
        tok.padding_side = "left"
        prompts = [f"質問: {ex['question']}\n選択肢: 1. {ex['choice0']} 2." for ex in val]
        comp = []
        for b in batches(len(val), 32):
            enc = tok([prompts[i] for i in b], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tok.pad_token_id)
            comp += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        tok.padding_side = "right"
        # 「 3. 」のような番号で区切る (選択肢に「3.5」のような数字があると誤って切れるが、JCommonsenseQA の選択肢にはない)
        got = [re.split(r"\s*\d\.\s*", " 2." + c.split("\n")[0])[1:5] for c in comp]
        n_match = np.array([sum(g == t for g, t in zip(gs, choices(ex)[1:])) for gs, ex in zip(got, val)])
        acc = {"元の問題": predict_number(model, tok, [mc_text(ex["question"], choices(ex)) for ex in val]) == gold,
               "問題文を言い換え": predict_number(model, tok, [mc_text(p, choices(ex)) for p, ex in zip(para, val)]) == gold,
               "選択肢を並べ替え": predict_number(model, tok, [mc_text(ex["question"], o) for ex, (o, _) in zip(val, shuffled)]) == np.array([l for _, l in shuffled])}
    scores = pd.DataFrame({
        "損失 (平均対数尤度)": [lp.mean() for lp in lps],
        "Min-20% Prob": [np.sort(lp)[:max(1, int(len(lp) * 0.2))].mean() for lp in lps],     # 確率の低いほうから 20% のトークンの平均
        "zlib 比": [lp.sum() / len(zlib.compress(t.encode())) for lp, t in zip(lps, texts)],  # 圧縮しやすい (ありふれた) 文の高い尤度を割り引く
        "続きの生成 (一致した選択肢の数)": n_match})
    return scores, acc, comp

def report(model, label, scores, acc):
    y = np.r_[np.ones(len(SEEN)), np.zeros(len(UNSEEN))]
    row = {f"AUC {c}": roc_auc_score(y, np.r_[scores[c].values[SEEN], scores[c].values[UNSEEN]]) for c in scores}
    row["生成で 4 つとも一致 SEEN/UNSEEN"] = f"{(scores.iloc[SEEN, 3] == 4).mean():.0%} / {(scores.iloc[UNSEEN, 3] == 4).mean():.0%}"
    for k, v in acc.items():
        row[f"正解率 {k} SEEN/UNSEEN"] = f"{v[SEEN].mean():.0%} / {v[UNSEEN].mean():.0%}"
    model.eval()
    row["並び順の検定の順位 SEEN/UNSEEN"] = " / ".join(f"{order_ranks(model, tok, FORMATS['素のテキスト'], idx).mean():.2f}" for idx in [SEEN, UNSEEN])
    results[label] = row

results = {}
scores, acc, comp = detect(model)
report(model, "追加学習の前", scores, acc)
print(pd.DataFrame(results).round(3).to_string())

                               追加学習の前
AUC 損失 (平均対数尤度)              0.519778
AUC Min-20% Prob             0.525467
AUC zlib 比                   0.499644
AUC 続きの生成 (一致した選択肢の数)        0.486489
生成で 4 つとも一致 SEEN/UNSEEN       0% / 0%
正解率 元の問題 SEEN/UNSEEN        61% / 61%
正解率 問題文を言い換え SEEN/UNSEEN    53% / 57%
正解率 選択肢を並べ替え SEEN/UNSEEN    57% / 57%
並び順の検定の順位 SEEN/UNSEEN     0.49 / 0.53


In [6]:
# 追加学習: 学習データ 600 問 + SEEN の 150 問を混ぜ、1 エポックずつ学習しては検出を試す (Colab の T4 で 4〜6 分)
texts_train = [mc_text(ex["question"], choices(ex), ex["label"]) for ex in train] + [mc_text(val[i]["question"], choices(val[i]), val[i]["label"]) for i in SEEN]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5, weight_decay=0.0)
scaler = torch.amp.GradScaler(enabled=not BF16)            # float16 の混合精度では勾配のスケーリングが要る
torch.cuda.reset_peak_memory_stats()                       # 以下のメモリ最大値は、追加学習と検出だけの値にする
for epoch in range(1, 4):
    model.train(); t = time.time()
    order = rng.permutation(len(texts_train))
    for b in batches(len(order), 16):
        enc = tok([texts_train[order[i]] for i in b], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        labels = enc["input_ids"].masked_fill(enc["attention_mask"] == 0, -100)
        with amp():
            loss = model(**enc, labels=labels).loss
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad()
    scores, acc, comp = detect(model)
    report(model, f"{epoch} エポック後", scores, acc)
    print(f"エポック {epoch}: 損失 {loss.item():.3f}, {time.time() - t:.0f} 秒, GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(pd.DataFrame(results).round(3).to_string())
print("\n3 エポック後の続きの生成の例 (SEEN):", choices(val[0])[1:], "→", comp[0].split("\n")[0])
print("同 (UNSEEN):", choices(val[150])[1:], "→", comp[150].split("\n")[0])

エポック 1: 損失 1.775, 25 秒, GPU メモリ最大 12.4 GB
エポック 2: 損失 1.374, 25 秒, GPU メモリ最大 12.4 GB
エポック 3: 損失 1.020, 25 秒, GPU メモリ最大 12.4 GB
                               追加学習の前      1 エポック後      2 エポック後      3 エポック後
AUC 損失 (平均対数尤度)              0.519778     0.854311     0.993022     0.999956
AUC Min-20% Prob             0.525467     0.806711     0.984089          1.0
AUC zlib 比                   0.499644     0.824311     0.988667     0.999956
AUC 続きの生成 (一致した選択肢の数)        0.486489     0.539711     0.608933     0.812756
生成で 4 つとも一致 SEEN/UNSEEN       0% / 0%      0% / 0%      0% / 0%     13% / 0%
正解率 元の問題 SEEN/UNSEEN        61% / 61%    85% / 77%    96% / 83%    96% / 83%
正解率 問題文を言い換え SEEN/UNSEEN    53% / 57%    83% / 78%    94% / 81%    93% / 81%
正解率 選択肢を並べ替え SEEN/UNSEEN    57% / 57%    83% / 78%    89% / 84%    93% / 83%
並び順の検定の順位 SEEN/UNSEEN     0.49 / 0.53  0.77 / 0.49  0.95 / 0.49  0.99 / 0.51

3 エポック後の続きの生成の例 (SEEN): ['小浜', '大浜', '小売', '高浜'] →  大浜 3. 小鍋町 4. 大浜海岸 5. 大浜
同 (UNSEEN): ['大西洋', '海峡', '

In [7]:
# 正解率の差は偶然の範囲か (3 エポック後)。SEEN と UNSEEN は別の問題なので、ここは対応のないブートストラップ
def unpaired_diff(a, b, n_boot=2000):
    d = [(rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean()) * 100 for _ in range(n_boot)]
    return f"{(a.mean() - b.mean()) * 100:+5.1f} [{np.percentile(d, 2.5):+5.1f}, {np.percentile(d, 97.5):+5.1f}]"
for k, v in acc.items():
    print(f"{k}: SEEN − UNSEEN {unpaired_diff(v[SEEN], v[UNSEEN])}")
for part, idx in [("SEEN", SEEN), ("UNSEEN", UNSEEN)]:
    print(f"{part}: 言い換え − 元 {paired_diff(acc['問題文を言い換え'][idx], acc['元の問題'][idx])}, 並べ替え − 元 {paired_diff(acc['選択肢を並べ替え'][idx], acc['元の問題'][idx])}")

元の問題: SEEN − UNSEEN +13.3 [ +6.7, +20.0]
問題文を言い換え: SEEN − UNSEEN +12.0 [ +4.7, +20.0]
選択肢を並べ替え: SEEN − UNSEEN +10.0 [ +2.7, +17.3]
SEEN: 言い換え − 元  -3.3 [ -7.3,  +0.0], 並べ替え − 元  -2.7 [ -6.7,  +1.3]
UNSEEN: 言い換え − 元  -2.0 [ -6.7,  +2.0], 並べ替え − 元  +0.7 [ -4.7,  +6.7]


In [8]:
# (3) 答えの分からない公開モデルに使える検定: 選択肢の元の並び順の尤度が、並べ替えた順より高いか (検定の関数は上で定義した)
tests = {}
# まず、上でわざと汚染させたモデルの SEEN / UNSEEN で試す (実際に学習した問題を見分けられるか)
tests["汚染させた 0.5b: SEEN"] = {f: order_test(model, tok, fmt, SEEN) for f, fmt in FORMATS.items()}
tests["汚染させた 0.5b: UNSEEN"] = {f: order_test(model, tok, fmt, UNSEEN) for f, fmt in FORMATS.items()}
del model, opt; gc.collect(); torch.cuda.empty_cache()

# 対照: JGLUE (2022 年) より前に公開されたモデル (rinna/japanese-gpt2-small、2021 年 8 月公開、日本語の CC-100 と Wikipedia で学習、MIT ライセンス)
name = "rinna/japanese-gpt2-small"
tok = AutoTokenizer.from_pretrained(name)
# transformers 5 系の T5Tokenizer は未知語を文末記号 (</s>) の ID にしてしまう。spiece.model から語彙とスコアを読んで Unigram に戻す
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "spiece.model"))
tok.backend_tokenizer.model = models.Unigram([(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id())
model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()
tests["rinna gpt2-small (JGLUE より前)"] = {f: order_test(model, tok, fmt, np.arange(300)) for f, fmt in FORMATS.items()}
del model; gc.collect(); torch.cuda.empty_cache()
print(pd.DataFrame(tests).T.to_string())

                                                         素のテキスト                              JSON
汚染させた 0.5b: SEEN              0.986 (p=4.8e-146, 最も尤度が高い割合 91%)  0.868 (p=1.5e-50, 最も尤度が高い割合 52%)
汚染させた 0.5b: UNSEEN                  0.506 (p=0.4, 最も尤度が高い割合 7%)      0.477 (p=0.85, 最も尤度が高い割合 3%)
rinna gpt2-small (JGLUE より前)       0.472 (p=0.96, 最も尤度が高い割合 6%)      0.517 (p=0.15, 最も尤度が高い割合 5%)


In [9]:
# 公開されている事前学習モデルそのもの (0.5b と 3b) で、評価データの 300 問を調べる (Colab の T4 で 3〜5 分)
for name in ["sbintuitions/sarashina2.2-0.5b", "sbintuitions/sarashina2.2-3b"]:
    tok = load_tokenizer(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE, device_map="auto").eval()
    tests[name.split("/")[-1]] = {f: order_test(model, tok, fmt, np.arange(300)) for f, fmt in FORMATS.items()}
    del model; gc.collect(); torch.cuda.empty_cache()
print(pd.DataFrame(tests).T.to_string())

                                                         素のテキスト                              JSON
汚染させた 0.5b: SEEN              0.986 (p=4.8e-146, 最も尤度が高い割合 91%)  0.868 (p=1.5e-50, 最も尤度が高い割合 52%)
汚染させた 0.5b: UNSEEN                  0.506 (p=0.4, 最も尤度が高い割合 7%)      0.477 (p=0.85, 最も尤度が高い割合 3%)
rinna gpt2-small (JGLUE より前)       0.472 (p=0.96, 最も尤度が高い割合 6%)      0.517 (p=0.15, 最も尤度が高い割合 5%)
sarashina2.2-0.5b                  0.503 (p=0.44, 最も尤度が高い割合 9%)      0.493 (p=0.67, 最も尤度が高い割合 5%)
sarashina2.2-3b                    0.506 (p=0.36, 最も尤度が高い割合 6%)      0.495 (p=0.61, 最も尤度が高い割合 4%)


- 汚染を調べる方法は大きく分けて、(a) 問題の続きを生成させて元の文と一致するかを見る、(b) 問題の文の尤度を見る、(c) 問題を言い換えたり選択肢を並べ替えたりして正解率が落ちるかを見る、の 3 つ。(b) の Min-K% Prob (Shi et al., 2023) は、見たことのない文には確率の極端に低いトークンが混じりやすいと考え、確率の低いほうから k% のトークンの平均対数尤度を使う。ここでは平均対数尤度 (損失) と、zlib で圧縮したバイト数で割った値も並べた。この章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」の注意書きにある汚染の問題を、実際に調べてみるためのレシピである。
- どの方法が効くかは、答えが分かっていないと確かめられない。そこで、評価データの 150 問 (SEEN) だけを学習データに混ぜて 0.5b のモデルを追加学習し、残りの 150 問 (UNSEEN) と見分けられるかを調べた。尤度にもとづく 3 つのスコアの ROC-AUC は、1 エポック (各問題を 1 回だけ学習) で 0.81〜0.85、3 エポックでほぼ 1.0 になった。この実行では、Min-20% Prob (0.807) は単純な損失 (0.854) より良くはなかった。続きの生成は尤度のスコアより弱く、AUC は 1 エポックで 0.540、3 エポックで 0.813 まで上がったが、4 つの選択肢をすべて再現できたのは 3 エポック後でも SEEN の 13% だった (例のように、似た語は出るが元のとおりではない)。生成で一致しないことは、見ていないことの根拠にならない。
- 汚染があると点数は高く出る。3 エポック後の正解率は SEEN 96%、UNSEEN 83% で、差は +13.3 ポイント [+6.7, +20.0] (対応のないブートストラップの 95% 信頼区間) だった。問題文を言い換えても、選択肢を並べ替えても SEEN の有利は残り (+12.0、+10.0 ポイント)、言い換えによる SEEN の正解率の変化 (−3.3 ポイント [−7.3, 0.0]) は UNSEEN (−2.0 ポイント) と区別がつかなかった。モデルは答えの番号だけでなく内容を覚えるので、「言い換えても点数が落ちない」ことは汚染がないことの証明にならない。実際のモデル (3b の指示チューニング済みモデル) でも、言い換えで −1.3 ポイント [−4.7, +2.0]、並べ替えで +1.3 ポイント [−3.0, +5.7] と落ちなかったが、同じ理由で汚染の有無は言えない。言い換えはモデルに作らせたので、意味が少しずれた問題も混じりうる (例の「よくわかるように述べることは？」)。
- 尤度にもとづく方法は、見ていないことが確かな、同じ種類の文と比べてはじめて意味を持つ。上の実験では UNSEEN がそれに当たるが、公開モデルではふつうそれがない。公開日の新しいデータと比べると、時期や話題の違い (分布のずれ) まで検出してしまう。Duan et al. (2024) は、事前学習データに対するこの種の判定 (メンバーシップ推論) が多くの設定で偶然とほとんど変わらず、うまくいくとされた設定の一部は、メンバーと非メンバーの時期の違いなどの分布のずれで説明できると報告している。
- 比べる相手が要らない方法として、Oren et al. (2023) の「元の並び順の尤度が、並べ替えた順より高いか」という考え方を、選択肢の順序に当てはめた。汚染させたモデルの SEEN では、順位の平均が 0.986 (p=4.8e-146) と元の並び順がほぼ常に最も尤度が高く、1 エポック後でも 0.77 だった。UNSEEN (0.506) と、JGLUE (Kurihara et al., 2022) より前に公開された rinna/japanese-gpt2-small (0.472、0.517) は 0.5 前後だった。公開されている sarashina2.2 の事前学習モデル (0.5b、3b) は 0.493〜0.506 (p≧0.36) で、元の並び順を好む証拠は見つからなかった。
- 何も検出されなくても、汚染がないとは言えない。ここでの汚染 (750 問だけを、各問題 1〜3 回、問題の形式のまま学習) は、事前学習で膨大なデータに混じって数回見るのよりずっと強い。また、モデルが見たかもしれない形 (Web ページ、データセットのビューア、翻訳されたもの、答え付きの解説など) と、検定に使う形が違うと検出しにくい。学習した形と違う JSON の形式では、汚染させたモデルでも「最も尤度が高い割合」が 91% から 52% に下がり、順位の平均も 0.986 から 0.868 に弱まった (この強さの汚染なら p=1.5e-50 でまだ検出できる)。並び順の検定は、データセットの選択肢の順序に意味がない (五十音順や作られた順のような規則で並んでいない) ことも前提にしている。sarashina2.2 のモデルカードには、3 段階で学習し、Web コーパスから抽出したデータ (10 兆トークン)、数学とコードの合成データ、各種タスク向けの少量のデータを使ったとあるが、データの中身は公開されておらず、評価データが含まれるかは書かれていない。
- 結論を強くするには、新しく作った問題や非公開の問題で点数が保たれるかを見るのがいちばん確かである。学習データが公開されているモデルなら、評価データと学習データを n-gram などで直接照合できる。ここで調べたのは評価データの 300 問だけで、JCommonsenseQA の学習データを指示チューニングに使ったかどうか (同じ形式の問題で有利になりうる) は確かめていない。
- データは JCommonsenseQA (CC BY-SA 4.0)、モデルは sarashina2.2 (SB Intuitions、MIT ライセンス) と rinna/japanese-gpt2-small (rinna、MIT ライセンス、日本語の CC-100 と Wikipedia で学習)。追加学習は語彙の埋め込みと出力層を固定し、GPU メモリを最大 12.4 GB 使った (追加学習と検出の間の最大値。T4 と同じ float16 の混合精度で実行した値)。sarashina2.2 のトークナイザーは、この章の「オープンな LLM とチャットする (チャットテンプレート)」の注意書きのとおりに直して使う。rinna のトークナイザーも、transformers 5 系では未知語の扱いが壊れるので、コードのとおりに直す。